# TruePin

In [51]:
import pandas as pd
import geopandas as gpd
from shapely import wkb
from shapely.geometry import box

df = pd.read_parquet("project_d_samples.parquet")
df.head()

,id,geometry,bbox,type,version,sources,names,categories,confidence,websites,socials,emails,phones,brand,addresses
0,08f266694b8dd34103db5d8f268c8bfe,b'\x01\x01\x00\x00\x00@]\xee\xdd0\rV\xc0\xdb\x...,"{'xmax': -88.20610046386719, 'xmin': -88.20610...",place,0,"[{'confidence': 0.8770555990602976, 'dataset':...","{'common': None, 'primary': 'SNT Biotech Lab',...","{'alternate': ['health_and_medical', 'laborato...",0.877056,[http://www.sntbiotech.com/],[https://www.facebook.com/109093534897634],None,[+13127662326],None,"[{'country': 'US', 'freeform': '24119 W Riverw..."
1,08f2af608b8633ac03a9430e8ba43b40,b'\x01\x01\x00\x00\x00\\\xef\xedLW$S\xc0)lPE;\...,"{'xmax': -76.56782531738281, 'xmin': -76.56784...",place,0,"[{'confidence': 0.804932735426009, 'dataset': ...","{'common': None, 'primary': 'Debbie’s Doula Se...","{'alternate': ['prenatal_perinatal_care'], 'pr...",0.804933,[http://debbiesdoulaservices.com/],[https://www.facebook.com/100596505513704],None,[+17572685658],None,"[{'country': 'US', 'freeform': '912 Arnett Dr'..."
2,08f44a112bcc56d30324dd2659dd9b1b,b'\x01\x01\x00\x00\x00\xc9v\xbe\x9f\x1a\x0cT\x...,"{'xmax': -80.18911743164062, 'xmin': -80.18912...",place,0,"[{'confidence': 0.9793990828827596, 'dataset':...","{'common': None, 'primary': 'Wax Custom Commun...","{'alternate': ['business_advertising', 'market...",0.995262,[http://www.waxcom.com],[https://www.facebook.com/106355339205],None,[+13053505700],None,"[{'country': 'US', 'freeform': '261 NE 1st St'..."
3,08f44dab0c4b02db035754f1e3be0658,b'\x01\x01\x00\x00\x00\x00-\xd6\x90U/T\xc0\xd9...,"{'xmax': -80.73958587646484, 'xmin': -80.73960...",place,0,"[{'confidence': 0.77, 'dataset': 'Microsoft', ...","{'common': None, 'primary': 'Advance Auto Part...","{'alternate': ['automotive', 'retail'], 'prima...",0.770000,[https://stores.advanceautoparts.com/nc/charlo...,None,None,[7045664047],None,"[{'country': 'US', 'freeform': '3020 Milton Rd..."
4,08f26e0cce79a7b603caea464ec624bb,b'\x01\x01\x00\x00\x00\xe0\x80\x96\xae\xe0XX\x...,"{'xmax': -97.38871002197266, 'xmin': -97.38872...",place,0,"[{'confidence': 0.9793990828827596, 'dataset':...","{'common': None, 'primary': 'Tanglefoot Wine C...","{'alternate': None, 'primary': 'winery'}",0.979399,[http://Tanglefootwinecellar.com/],[https://www.facebook.com/101890712147590],None,[+17855772944],None,"[{'country': 'US', 'freeform': '9925 S Amos Rd..."


In [52]:
df["geometry"] = df["geometry"].apply(wkb.loads)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

gdf.geometry.type.value_counts()

Point    3425
Name: count, dtype: int64

In [53]:
gdf[["geometry", "bbox"]].head()

,geometry,bbox
0,POINT (-88.20611 41.61867),"{'xmax': -88.20610046386719, 'xmin': -88.20610..."
1,POINT (-76.56783 37.13462),"{'xmax': -76.56782531738281, 'xmin': -76.56784..."
2,POINT (-80.18912 25.77545),"{'xmax': -80.18911743164062, 'xmin': -80.18912..."
3,POINT (-80.7396 35.23658),"{'xmax': -80.73958587646484, 'xmin': -80.73960..."
4,POINT (-97.38871 38.656),"{'xmax': -97.38871002197266, 'xmin': -97.38872..."


In [54]:
def bbox_to_polygon(b):
    return box(b["xmin"], b["ymin"], b["xmax"], b["ymax"])

gdf["bbox_geom"] = gdf["bbox"].apply(bbox_to_polygon)

In [55]:
gdf["bbox_geom"] = gpd.GeoSeries(gdf["bbox_geom"], crs=gdf.crs)
gdf.geometry.within(gdf["bbox_geom"]).value_counts()

True     3416
False       9
Name: count, dtype: int64

In [56]:
gdf.loc[~gdf.geometry.within(gdf["bbox_geom"]), 
        ["geometry", "bbox", "categories", "confidence"]].head(10)


,geometry,bbox,categories,confidence
1005,POINT (-73.97113 40.89867),"{'xmax': -73.97113037109375, 'xmin': -73.97113...","{'alternate': None, 'primary': 'personal_injur...",0.77
1028,POINT (-106.63314 35.04987),"{'xmax': -106.63314056396484, 'xmin': -106.633...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77
2123,POINT (-76.55529 38.78493),"{'xmax': -76.55529022216797, 'xmin': -76.55529...","{'alternate': ['fishing_charter', 'active_life...",0.77
2160,POINT (-80.14058 25.79691),"{'xmax': -80.14057922363281, 'xmin': -80.14057...","{'alternate': None, 'primary': 'public_relatio...",0.77
2204,POINT (-93.86113 32.44955),"{'xmax': -93.86112976074219, 'xmin': -93.86112...","{'alternate': ['hotel'], 'primary': 'resort'}",0.77
2337,POINT (-122.38413 40.59862),"{'xmax': -122.3841323852539, 'xmin': -122.3841...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77
2669,POINT (-97.15608 33.38482),"{'xmax': -97.15608215332031, 'xmin': -97.15608...","{'alternate': None, 'primary': 'septic_services'}",0.77
2670,POINT (-96.97173 32.99805),"{'xmax': -96.97172546386719, 'xmin': -96.97174...","{'alternate': ['electrician', 'utility_service...",0.77
2807,POINT (-117.80508 44.78466),"{'xmax': -117.8050765991211, 'xmin': -117.8050...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77


In [57]:
outside = gdf[~gdf.geometry.within(gdf["bbox_geom"])]

outside[[
    "id",
    "geometry",
    "bbox",
    "categories",
    "confidence",
    "sources"
]].head(10)


,id,geometry,bbox,categories,confidence,sources
1005,08f2a100a2c919110359c72481d422a2,POINT (-73.97113 40.89867),"{'xmax': -73.97113037109375, 'xmin': -73.97113...","{'alternate': None, 'primary': 'personal_injur...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
1028,08f48d894eab411e03eaad1173748485,POINT (-106.63314 35.04987),"{'xmax': -106.63314056396484, 'xmin': -106.633...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2123,08f2aa83b69b3b0b03445c22e8bd74ec,POINT (-76.55529 38.78493),"{'xmax': -76.55529022216797, 'xmin': -76.55529...","{'alternate': ['fishing_charter', 'active_life...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2160,08f44a111db4344103da386c1de44278,POINT (-80.14058 25.79691),"{'xmax': -80.14057922363281, 'xmin': -80.14057...","{'alternate': None, 'primary': 'public_relatio...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2204,08f444cad1332392030b4695218ddd7f,POINT (-93.86113 32.44955),"{'xmax': -93.86112976074219, 'xmin': -93.86112...","{'alternate': ['hotel'], 'primary': 'resort'}",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2337,08f28157656aba86031e5fe6a01507b5,POINT (-122.38413 40.59862),"{'xmax': -122.3841323852539, 'xmin': -122.3841...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2669,08f26c8553a63b0a0345288eac813b3f,POINT (-97.15608 33.38482),"{'xmax': -97.15608215332031, 'xmin': -97.15608...","{'alternate': None, 'primary': 'septic_services'}",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2670,08f26c8605c4a6f203fd07492cee6d87,POINT (-96.97173 32.99805),"{'xmax': -96.97172546386719, 'xmin': -96.97174...","{'alternate': ['electrician', 'utility_service...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."
2807,08f2881aa5bb639a03cf4d0483216ead,POINT (-117.80508 44.78466),"{'xmax': -117.8050765991211, 'xmin': -117.8050...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77,"[{'confidence': 0.77, 'dataset': 'Microsoft', ..."


## Ground truth labeling (faster manual workflow)

Because ground truth must be established manually (e.g. by checking each place on a map), the steps below help you do it much faster:

1. **Export a labeling worksheet** — 1000 places split into **5 CSV files of 200 each** (`ground_truth_labeling_1.csv` … `ground_truth_labeling_5.csv`) with name, address, current coordinates, and **one-click map links** (OpenStreetMap + optional Google Maps search) so you can verify quickly.
2. **Fill in labels** — In the CSV/Excel: set `label` to `accurate` / `inaccurate` / `skip`, and optionally `ground_truth_lon`, `ground_truth_lat` when you correct the pin.
3. **Load back** — Merge your filled worksheet into the main dataset to get a ground-truth subset.

You can either fill the CSV/Excel in bulk (e.g. open map link → decide → type label) or use the **interactive labeling UI** in the next section to label one-by-one in the notebook with progress saved to CSV.

In [58]:
# Build ground-truth labeling worksheet from gdf (run after the cells above that create gdf)
def _safe_primary_name(names):
    if names is None or not isinstance(names, dict):
        return ""
    return names.get("primary") or ""

def _safe_freeform_address(addresses):
    if not addresses or not isinstance(addresses, list):
        return ""
    a = addresses[0]
    return (a.get("freeform") or "") if isinstance(a, dict) else ""

def build_labeling_worksheet(gdf, out_path="ground_truth_labeling.csv"):
    """Build a CSV with one row per place: id, name, address, coords, map links, and empty label columns."""
    rows = []
    for _, row in gdf.iterrows():
        geom = row["geometry"]
        lon, lat = float(geom.x), float(geom.y)
        name = _safe_primary_name(row.get("names"))
        address = _safe_freeform_address(row.get("addresses"))
        # One-click map links (no API key)
        osm_link = f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}&zoom=17"
        gmaps_search = f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"
        if address:
            gmaps_search = f"https://www.google.com/maps/search/?api=1&query={address}"
        rows.append({
            "id": row["id"],
            "name": name,
            "address": address,
            "lon": lon,
            "lat": lat,
            "map_osm": osm_link,
            "map_google": gmaps_search,
            "ground_truth_lon": "",
            "ground_truth_lat": "",
            "label": "",  # accurate | inaccurate | skip
        })
    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False)
    print(f"Exported {len(out)} rows to {out_path}")
    return out

# Ground truth: 1000 places split into 5 files of 200 each
GROUND_TRUTH_SAMPLE_SIZE = 1000
CHUNK_SIZE = 200
subset = gdf.sample(n=min(GROUND_TRUTH_SAMPLE_SIZE, len(gdf)), random_state=42)
ground_truth_filenames = []
for k in range(1, 6):
    start = (k - 1) * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(subset))
    chunk = subset.iloc[start:end]
    fname = f"ground_truth_labeling_{k}.csv"
    build_labeling_worksheet(chunk, fname)
    ground_truth_filenames.append(fname)
print(f"Created 5 files: {ground_truth_filenames}")
labeling_df = pd.read_csv(ground_truth_filenames[0])  # for reference; UI loads by filename
labeling_df.head()

Exported 200 rows to ground_truth_labeling_1.csv
Exported 200 rows to ground_truth_labeling_2.csv
Exported 200 rows to ground_truth_labeling_3.csv
Exported 200 rows to ground_truth_labeling_4.csv
Exported 200 rows to ground_truth_labeling_5.csv
Created 5 files: ['ground_truth_labeling_1.csv', 'ground_truth_labeling_2.csv', 'ground_truth_labeling_3.csv', 'ground_truth_labeling_4.csv', 'ground_truth_labeling_5.csv']


,id,name,address,lon,lat,map_osm,map_google,ground_truth_lon,ground_truth_lat,label
0,08f28d5428cd954e0338fdd58d90561a,La Belle Elaine's Bridal,NaN,-122.325373,47.633725,https://www.openstreetmap.org/?mlat=47.6337251...,https://www.google.com/maps/search/?api=1&quer...,NaN,NaN,NaN
1,08f26e8070c2d6a9033d1fa93bda0a8f,Thunderbowl,NaN,-94.981920,35.914160,https://www.openstreetmap.org/?mlat=35.91416&m...,https://www.google.com/maps/search/?api=1&quer...,NaN,NaN,NaN
2,08f48981360c349403c80ceff2683b6b,Hofbrau RV Park,NaN,-98.331503,30.460491,https://www.openstreetmap.org/?mlat=30.4604909...,https://www.google.com/maps/search/?api=1&quer...,NaN,NaN,NaN
3,08f29a0b5b4a978203d20beb95a197d2,Masterlink Sausage and Meats,NaN,-117.890272,33.873189,https://www.openstreetmap.org/?mlat=33.8731889...,https://www.google.com/maps/search/?api=1&quer...,NaN,NaN,NaN
4,08f28f00f3d4b585032f78576978daf6,Additional Self Storage,NaN,-122.621135,45.667548,https://www.openstreetmap.org/?mlat=45.6675478...,https://www.google.com/maps/search/?api=1&quer...,NaN,NaN,NaN


In [59]:
# After you fill in the CSV (label and optionally ground_truth_lon/lat), load it back and merge
def load_ground_truth(csv_path="ground_truth_labeling.csv"):
    """Load your filled labeling CSV and return a DataFrame with ground truth columns."""
    filled = pd.read_csv(csv_path)
    # Normalize label column (strip, lowercase)
    if "label" in filled.columns:
        filled["label"] = filled["label"].astype(str).str.strip().str.lower()
    # Parse numeric ground truth coords (empty stays NaN)
    for col in ("ground_truth_lon", "ground_truth_lat"):
        if col in filled.columns:
            filled[col] = pd.to_numeric(filled[col], errors="coerce")
    return filled

# Example: load one file or concatenate all 5
# filled = load_ground_truth("ground_truth_labeling_1.csv")
# all_filled = pd.concat([load_ground_truth(f) for f in ground_truth_filenames], ignore_index=True)
# labeled = all_filled[all_filled["label"].isin(["accurate", "inaccurate"])]
# ground_truth_gdf = gdf.merge(labeled[["id", "label", "ground_truth_lon", "ground_truth_lat"]], on="id", how="inner")

### Interactive labeling (one-by-one with save)

Run the cell below to label places one-by-one. **Set `LABEL_CSV`** in the code box to the file you want to edit (e.g. `ground_truth_labeling_3.csv` for file 3). Use **Accurate** / **Inaccurate** / **Skip** to save each row; you can re-run the cell with a different filename to work on another of the 5 files.

In [60]:
# Interactive labeling UI (optional: install with pip install ipywidgets if needed)
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
except ImportError:
    print("Install ipywidgets for the interactive UI: pip install ipywidgets")
    raise

# Which file to edit: set to ground_truth_labeling_1.csv ... ground_truth_labeling_5.csv
LABEL_CSV = "ground_truth_labeling_1.csv"

import os
if not os.path.exists(LABEL_CSV):
    raise FileNotFoundError(f"{LABEL_CSV} not found. Run the worksheet cell above first to create the 5 files (ground_truth_labeling_1.csv ... ground_truth_labeling_5.csv).")
_df = pd.read_csv(LABEL_CSV, dtype={"label": str})
# Keep ground truth coords as float64 (NaN when not set)
_df["ground_truth_lon"] = pd.to_numeric(_df["ground_truth_lon"], errors="coerce")
_df["ground_truth_lat"] = pd.to_numeric(_df["ground_truth_lat"], errors="coerce")

idx = widgets.IntSlider(min=0, max=max(0, len(_df) - 1), step=1, value=0, description="Index")
info = widgets.HTML(value="")
# Use Text to avoid NaN (not JSON-serializable in Jupyter)
gt_lon = widgets.Text(value="", description="GT lon", placeholder="optional")
gt_lat = widgets.Text(value="", description="GT lat", placeholder="optional")
# Paste "lat, lon" or "lon, lat" (comma/space/tab/newline separated) and click Fill
paste_coords = widgets.Text(value="", placeholder="Paste coords: e.g. 41.618, -88.206 or -88.206 41.618", layout=widgets.Layout(width="400px"))
fill_btn = widgets.Button(description="Fill from paste")

def update_display(index):
    if index < 0 or index >= len(_df):
        info.value = "No row at this index."
        return
    r = _df.iloc[index]
    name = r.get("name", "")
    address = r.get("address", "")
    lon, lat = r.get("lon"), r.get("lat")
    osm = r.get("map_osm", "")
    gl = r.get("map_google", "")
    lbl = r.get("label", "")
    info.value = f"""
    <b>Place {index + 1} / {len(_df)}</b><br>
    <b>Name:</b> {name}<br>
    <b>Address:</b> {address}<br>
    <b>Current pin:</b> {lon}, {lat}<br>
    <b>Your label:</b> {lbl}<br>
    <a href="{osm}" target="_blank">Open in OSM</a> &nbsp;
    <a href="{gl}" target="_blank">Open in Google Maps</a>
    """
    lo, la = r.get("ground_truth_lon"), r.get("ground_truth_lat")
    gt_lon.value = str(lo).strip() if pd.notna(lo) and str(lo).strip() else ""
    gt_lat.value = str(la).strip() if pd.notna(la) and str(la).strip() else ""

def _parse_opt_float(s):
    s = (s or "").strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None

def _parse_pasted_coords(s):
    """Parse pasted text into (lon, lat). Accepts 'lat,lon', 'lon,lat', or space/tab/newline separated."""
    import re
    s = (s or "").strip()
    if not s:
        return None, None
    # Split on comma, space, tab, or newline and extract numbers
    parts = re.split(r"[, \t\n]+", s)
    nums = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        try:
            nums.append(float(p))
        except ValueError:
            continue
    if len(nums) < 2:
        return None, None
    a, b = nums[0], nums[1]
    # Infer order: lat in [-90,90], lon in [-180,180]
    if -90 <= a <= 90 and -180 <= b <= 180:
        return b, a  # lon, lat
    if -180 <= a <= 180 and -90 <= b <= 90:
        return a, b  # lon, lat
    return b, a  # assume first is lat, second lon

def on_fill_paste(btn):
    lon, lat = _parse_pasted_coords(paste_coords.value)
    if lon is not None and lat is not None:
        gt_lon.value = str(lon)
        gt_lat.value = str(lat)
        print(f"Filled GT lon={lon}, lat={lat}")
    else:
        print("Could not parse two numbers. Use e.g. 41.618, -88.206 or -88.206 41.618")

fill_btn.on_click(on_fill_paste)

def on_label(btn):
    i = idx.value
    if i < 0 or i >= len(_df):
        return
    r = _df.iloc[i]
    _df.at[_df.index[i], "label"] = btn.description
    if btn.description == "Accurate":
        # Ground truth = current position
        _df.at[_df.index[i], "ground_truth_lon"] = float(r["lon"])
        _df.at[_df.index[i], "ground_truth_lat"] = float(r["lat"])
    else:
        lo, la = _parse_opt_float(gt_lon.value), _parse_opt_float(gt_lat.value)
        if lo is not None and la is not None:
            _df.at[_df.index[i], "ground_truth_lon"] = lo
            _df.at[_df.index[i], "ground_truth_lat"] = la
        else:
            _df.at[_df.index[i], "ground_truth_lon"] = float("nan")
            _df.at[_df.index[i], "ground_truth_lat"] = float("nan")
    _df.to_csv(LABEL_CSV, index=False)
    idx.value = min(i + 1, len(_df) - 1)
    update_display(idx.value)
    print(f"Saved '{btn.description}' for index {i} -> CSV; moved to index {idx.value}")

def on_idx_change(change):
    update_display(change["new"])

idx.observe(on_idx_change, names="value")
update_display(0)

btns = [widgets.Button(description=lbl) for lbl in ("Accurate", "Inaccurate", "Skip")]
for b in btns:
    b.on_click(on_label)

display(widgets.VBox([
    widgets.HBox([idx, *btns]),
    info,
    widgets.HBox([gt_lon, gt_lat]),
    widgets.HBox([paste_coords, fill_btn]),
    widgets.HTML(value="<small>Paste coords (e.g. 41.618, -88.206 or -88.206 41.618) → Fill from paste → then click Accurate/Inaccurate/Skip to save.</small>"),
]))